# Task #3.0 : Decision trees
Implement a Decision Tree to classify the localization site of proteins (the name attribute)  in the given samples from yeast.csv and evaluate performance using AUC, F1 Score, Matthews Correlation Coefficient (MCC) and recall rate.

Step 1 :
Import all necessary libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Preprocessing
from sklearn.preprocessing import LabelEncoder, label_binarize

# Metrics & validation
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import f1_score, recall_score, matthews_corrcoef, roc_auc_score

Step 2 : Load the data from the csv file 

In [ ]:
data = pd.read_csv('yeast.csv')
features = ['mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc']
X = data[features].values
Y = data['name'].values
print(f"X shape : {X.shape}")
print(f"Y shape : {Y.shape}")

Step 3 : Check for class imbalance and encode target (Y)

Result : Heavily imbalanced classes
     
CYT - 463       
NUC - 429         
MIT - 244        
ME3 - 163    
ME2 - 51        
ME1 - 44       
EXC - 35         
VAC - 30        
POX - 20            
ERL - 5           - [ Very low!!!(Also why number of folds = 5) ]

In [ ]:
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

data['name'].value_counts()

Step 4 : Finding best hyperparameters for decision tree (Regularisation) as well as best score using GridSearchCV

Results : 

Best hyperparameters - 

class weight : balanced      
criterion: entropy      
max depth : 4    
min_samples_leaf : 1    
min_samples_split : 20    

Best CV F1 macro - 0.47816214656472084

In [ ]:
param_grid = {
    'max_depth': [4, 8, 12, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    DecisionTreeClassifier(),
    param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid.fit(X, Y)

print("Best hyperparameters:", grid.best_params_)
print("Best CV F1 macro:", grid.best_score_)

Step 5 : Actual Prediction using Stratified K-Fold (With best hyperparameters and Cross-Validation)

In [ ]:
best_clf = DecisionTreeClassifier(
    class_weight='balanced',
    criterion='entropy',
    max_depth=4,
    min_samples_leaf=1,
    min_samples_split=20
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Y_pred = cross_val_predict(best_clf, X, Y_encoded, cv=cv)

Step 6 : Evaluating performance using metrics like AUC, F1 Score, Matthews Correlation Coefficient (MCC) and recall rate

In [ ]:
Y_pred = cross_val_predict(best_clf, X, Y_encoded, cv=cv, method ='predict')
f1 = f1_score(Y_encoded, Y_pred, average='macro')
recall = recall_score(Y_encoded, Y_pred, average='macro')
mcc = matthews_corrcoef(Y_encoded, Y_pred)

Y_proba = cross_val_predict(best_clf, X, Y_encoded, cv=cv, method='predict_proba')
Y_bin = label_binarize(Y_encoded, classes=np.unique(Y_encoded))
auc_macro = roc_auc_score(Y_bin, Y_proba, average='macro', multi_class='ovr')

print("F1 (macro):", f1)
print("Recall (macro):", recall)
print("MCC:", mcc)
print("AUC (macro, OVR):", auc_macro)

# Results :      
F1 (macro) : 0.4915798074144225          
Recall (macro) : 0.5383094269472671       
MCC : 0.33851140253589146      
AUC (macro, OVR) : 0.7766736125395033       

This is the upper limit of what a single decision tree can do in this case. While still lacking accuracy, it is pretty balanced across classes and generalizes well.

Step 7 : Plotting Decision tree for better understanding

In [ ]:
best_clf = grid.best_estimator_

plt.figure(figsize=(30, 18))
plot_tree(best_clf,
          filled=True,
          rounded=True,
          fontsize=6,
          feature_names=features,
          class_names=best_clf.classes_)

plt.title("Decision Tree (Best Hyperparameters)")
plt.show()


Step 8 : Checking performance with reference to other models        

| Model              | F1 Macro | Recall | MCC    | AUC   |
|--------------------|----------|--------|--------|--------|
| Decision Tree      | 0.4727   | 0.5384 | 0.3759 | 0.7802 |
| Random Forest      | 0.5389   | 0.5384 | 0.4970 | 0.8727 |
| Gradient Boosting  | 0.5311   | 0.5325 | 0.4812 | 0.8708 |
| AdaBoost           | 0.2893   | 0.3087 | 0.3503 | 0.8287 |       

As expected, random forest works best since it reduces variance and handles small classes better,          
Gradient boosting performs similarly except struggling with small minority classes, noisy labels, non-tuned learning rate,       
AdaBoost performs poorly since it is extremely sensitive to noisy or tiny classes.

In [ ]:
# Encode labels
le = LabelEncoder()
Y_enc = le.fit_transform(Y)

# Define models
models = {
    "Decision Tree (Best)": DecisionTreeClassifier(
        class_weight='balanced',
        criterion='entropy',
        max_depth=4,
        min_samples_leaf=1,
        min_samples_split=20
    ),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier()
}

# AUC helper
def auc_macro_ovr(model, X_test, y_test):
    classes = np.unique(y_test)
    y_bin = label_binarize(y_test, classes=classes)
    y_proba = model.predict_proba(X_test)
    return roc_auc_score(y_bin, y_proba, average="macro")


# ---- 5-FOLD CV ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {name: [] for name in models}

for train_idx, test_idx in skf.split(X, Y_enc):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = Y_enc[train_idx], Y_enc[test_idx]

    for name, model in models.items():
        # Fit model ON THIS FOLD (this was missing before)
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Metrics
        f1 = f1_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        mcc = matthews_corrcoef(y_test, y_pred)

        # AUC (works now because model is fitted)
        auc = auc_macro_ovr(model, X_test, y_test)

        results[name].append((f1, recall, mcc, auc))


# ---- PRINT RESULTS ----
print("=== 5-FOLD STRATIFIED CV RESULTS ===")

for name, scores in results.items():
    f1 = np.mean([s[0] for s in scores])
    recall = np.mean([s[1] for s in scores])
    mcc = np.mean([s[2] for s in scores])
    auc = np.mean([s[3] for s in scores])

    print(f"\n{name}")
    print(f" F1 Macro:\t{f1:.4f}")
    print(f" Recall:\t{recall:.4f}")
    print(f" MCC:\t\t{mcc:.4f}")
    print(f" AUC Macro:\t{auc:.4f}")